# GEE Index Timelapse — Colab

Build a cloud-masked, monthly-median satellite animation over a forest AOI in a **few lines**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwinkelmann/GEE_Animation/blob/feat/evi-index/notebooks/colab_gee_animation.ipynb)

Steps: **install → authenticate once → load an AOI → `animate(...)`**. Makes live Earth Engine calls.

## 1 · Install (Colab only)

In [ ]:
import sys
IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    !pip install -q "git+https://github.com/cwinkelmann/GEE_Animation.git@feat/evi-index#egg=gee_animation[shapefile]"
# after the branch is merged, switch @feat/evi-index -> @main

## 2 · Authenticate Earth Engine (once)
A separate step — run it a single time. In Colab it opens an `ee.Authenticate()` prompt.

In [ ]:
from gee_animation import auth

PROJECT = "hnee-331218"   # <-- CHANGE THIS to your own Earth Engine project id
auth.init(PROJECT)

## 3 · Load an area of interest
Here we download the example **WNE forest** AOI from the repo and load it from disk. `animate()` reads the GeoJSON straight from the path.

In [ ]:
import urllib.request

AOI_URL = "https://raw.githubusercontent.com/cwinkelmann/GEE_Animation/feat/evi-index/docs/aoi/wne/wne.geojson"
urllib.request.urlretrieve(AOI_URL, "wne.geojson")   # download to the working dir
REGION = "wne.geojson"                                # load from disk

# --- Use your OWN AOI instead of the line above: ---
#   • upload a file in Colab:
#       from google.colab import files; files.upload()
#       REGION = "my_aoi.geojson"
#   • or point at any local GeoJSON path:  REGION = "/content/my_aoi.geojson"
#   • or a bounding box (no file needed):  REGION = [13.8, 52.9, 14.0, 53.05]  # [W, S, E, N]

## 4 · Choose products & dates

In [ ]:
# (sensor, index) pairs. Valid: sentinel2/landsat/modis + ndvi/evi/ndwi/ndmi/rgb/cir;
# landsat + lst/lst_smw/lst_sharp; modis_lst + lst_modis.
PRODUCTS = [("landsat", "lst"), ("sentinel2", "ndvi")]
START, END = "2023-01-01", "2024-01-01"
BUFFER_M, PRESET = 1000, "1080p"   # PRESET upscales native imagery so LST/MODIS are not tiny

## 5 · Generate & show
`animate()` does AOI → build → monthly median → render, and plays inline.

In [ ]:
from gee_animation import animate
from IPython.display import display

anims = []
for sensor, index in PRODUCTS:
    anim = animate(REGION, sensor=sensor, index=index, start=START, end=END,
                   buffer_m=BUFFER_M, preset=PRESET)
    display(anim)          # inline video
    anims.append(anim)

## 6 · Download (Colab)

In [ ]:
if IS_COLAB:
    from google.colab import files
    for a in anims:
        if a.mp4:
            files.download(a.mp4)